In [17]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif

In [2]:
adult_census_income_data=pd.read_csv("adult.csv")

In [3]:
adult_census_income_data.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [4]:
adult_census_income_data.isnull().sum()

age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64

In [5]:
adult_census_income_data['workclass'].value_counts()

workclass
Private             22696
Self-emp-not-inc     2541
Local-gov            2093
?                    1836
State-gov            1298
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64

In [6]:
adult_census_income_data['workclass'].mode()

0    Private
Name: workclass, dtype: str

In [7]:
adult_census_income_data['workclass']=adult_census_income_data['workclass'].replace('?','Private')

In [8]:
adult_census_income_data['occupation'].value_counts()

occupation
Prof-specialty       4140
Craft-repair         4099
Exec-managerial      4066
Adm-clerical         3770
Sales                3650
Other-service        3295
Machine-op-inspct    2002
?                    1843
Transport-moving     1597
Handlers-cleaners    1370
Farming-fishing       994
Tech-support          928
Protective-serv       649
Priv-house-serv       149
Armed-Forces            9
Name: count, dtype: int64

In [9]:
adult_census_income_data['occupation'].mode()

0    Prof-specialty
Name: occupation, dtype: str

In [10]:
adult_census_income_data['occupation']=adult_census_income_data['occupation'].replace('?','Prof-specialty')

In [11]:
adult_census_income_data.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,Private,77053,HS-grad,9,Widowed,Prof-specialty,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,Private,186061,Some-college,10,Widowed,Prof-specialty,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [22]:
adult_census_income_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [33]:
from sklearn.preprocessing import LabelEncoder

X = adult_census_income_data.drop('income', axis=1)
y = adult_census_income_data['income']

# Encode target
y = y.map({'<=50K':0, '>50K':1})

# Encode categorical features
cat_cols = X.select_dtypes(include='str').columns

le = LabelEncoder()
for col in cat_cols:
    X[col] = le.fit_transform(X[col])

x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=0)

# Compute MI
mi = mutual_info_classif(x_train, y_train, discrete_features=True)

# View scores
mi_series = pd.Series(mi, index=X.columns).sort_values(ascending=False)
print(mi_series)

fnlwgt            0.422510
relationship      0.115363
marital.status    0.109033
capital.gain      0.083500
age               0.068839
education         0.066526
education.num     0.066526
occupation        0.051466
hours.per.week    0.043756
capital.loss      0.037463
sex               0.025397
workclass         0.012357
native.country    0.006219
race              0.005614
dtype: float64


In [173]:
from sklearn.feature_selection import SelectKBest
sel_ten_cols=SelectKBest(mutual_info_classif, k=7)
sel_ten_cols.fit(x_train, y_train)
# 1. Get the list of selected column names
selected_cols = x_train.columns[sel_ten_cols.get_support()]

# 2. Filter x_train and x_test to retain only these columns
x_train_filtered = x_train[selected_cols]
x_test_filtered = x_test[selected_cols]

# Optional: Verify the new shapes
print(x_train_filtered.shape) # Output: (n_samples, 10)


(22792, 7)


In [174]:
x_train_filtered.columns

Index(['age', 'education', 'education.num', 'marital.status', 'occupation',
       'relationship', 'capital.gain'],
      dtype='str')

In [175]:
x_train_filtered.head()

,age,education,education.num,marital.status,occupation,relationship,capital.gain
32098,40,9,13,2,3,5,0
25206,39,11,9,2,6,0,0
23491,42,15,10,4,3,1,0
12367,27,11,9,4,4,3,0
7054,38,12,14,2,3,0,0


In [176]:
from sklearn.linear_model import LogisticRegression

model=LogisticRegression(max_iter=10000,C=0.1,class_weight={0:1, 1:2})

In [177]:
model.fit(x_train_filtered,y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.1
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*","{0: 1, 1: 2}"
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For 

In [178]:
y_preds=model.predict(x_test_filtered)

In [179]:
y_pred_probab=model.predict_proba(x_test_filtered)
y_pred_probab

array([[0.9850531 , 0.0149469 ],
       [0.95116137, 0.04883863],
       [0.8588901 , 0.1411099 ],
       ...,
       [0.91407045, 0.08592955],
       [0.94311399, 0.05688601],
       [0.45415603, 0.54584397]], shape=(9769, 2))

In [180]:
from sklearn.metrics import f1_score

f1_score=f1_score(y_test,y_preds,average='binary')

In [181]:
f1_score

0.5730431142737548

In [182]:
adult_census_income_data['income'].value_counts()

income
<=50K    24720
>50K      7841
Name: count, dtype: int64

In [183]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_preds))

              precision    recall  f1-score   support

           0       0.87      0.86      0.86      7410
           1       0.57      0.58      0.57      2359

    accuracy                           0.79      9769
   macro avg       0.72      0.72      0.72      9769
weighted avg       0.79      0.79      0.79      9769



In [184]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test,y_preds)

array([[6360, 1050],
       [ 990, 1369]])

In [209]:
y_prob=y_pred_probab[:,1]
treshold=0.7
y_pred_custom=(y_prob>=treshold).astype(int)

In [210]:
print(classification_report(y_test, y_pred_custom))

              precision    recall  f1-score   support

           0       0.82      0.95      0.88      7410
           1       0.71      0.36      0.48      2359

    accuracy                           0.81      9769
   macro avg       0.77      0.66      0.68      9769
weighted avg       0.80      0.81      0.79      9769



In [203]:
for t in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9]:
    y_pred = (y_prob >= t).astype(int)
    p = precision_score(y_test, y_pred)
    r = recall_score(y_test, y_pred)
    print(f"{t} → Precision: {p:.3f}, Recall: {r:.3f}")

0.5 → Precision: 0.566, Recall: 0.580
0.6 → Precision: 0.644, Recall: 0.475
0.7 → Precision: 0.712, Recall: 0.359
0.8 → Precision: 0.795, Recall: 0.243
0.85 → Precision: 0.879, Recall: 0.193
0.9 → Precision: 0.934, Recall: 0.156


In [204]:
from sklearn.metrics import precision_score, recall_score

for t in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9]:
    y_pred = (y_prob >= t).astype(int)
    p = precision_score(y_test, y_pred)
    r = recall_score(y_test, y_pred)
    print(f"{t} → Precision: {p:.3f}, Recall: {r:.3f}")

0.5 → Precision: 0.566, Recall: 0.580
0.6 → Precision: 0.644, Recall: 0.475
0.7 → Precision: 0.712, Recall: 0.359
0.8 → Precision: 0.795, Recall: 0.243
0.85 → Precision: 0.879, Recall: 0.193
0.9 → Precision: 0.934, Recall: 0.156


In [254]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train_scaled= scaler.fit_transform(x_train_filtered)
x_test_scaled= scaler.transform(x_test_filtered)

In [439]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=300, random_state=42,class_weight='balanced',min_samples_split=5,
                             min_samples_leaf=3,max_features='log2',max_depth=8)
clf.fit(x_train_scaled, y_train)

# Predict and evaluate
y_pred = clf.predict(x_test_scaled)

In [440]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.76      0.84      7410
           1       0.54      0.86      0.66      2359

    accuracy                           0.79      9769
   macro avg       0.74      0.81      0.75      9769
weighted avg       0.85      0.79      0.80      9769



In [441]:
y_pred_prob=clf.predict_proba(x_test_scaled)
y_prob1=y_pred_prob[:,1]
treshold=0.7
y_pred_custom=(y_prob1>=treshold).astype(int)

In [442]:
print(classification_report(y_test, y_pred_custom))

              precision    recall  f1-score   support

           0       0.88      0.93      0.90      7410
           1       0.73      0.61      0.66      2359

    accuracy                           0.85      9769
   macro avg       0.80      0.77      0.78      9769
weighted avg       0.84      0.85      0.85      9769



In [434]:
from sklearn.model_selection import RandomizedSearchCV

params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rf = RandomForestClassifier(class_weight='balanced', random_state=42)

search = RandomizedSearchCV(
    rf,
    param_distributions=params,
    n_iter=10,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

search.fit(x_train_filtered, y_train)

best_model = search.best_estimator_

In [231]:
best_model

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",4
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'log2'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metri

In [244]:
y_prob = best_model.predict_proba(x_test_filtered)[:,1]
treshold=0.7
y_pred_custom1=(y_prob>=treshold).astype(int)

In [245]:
print(classification_report(y_test,y_pred_custom1))

              precision    recall  f1-score   support

           0       0.88      0.92      0.90      7410
           1       0.72      0.61      0.66      2359

    accuracy                           0.85      9769
   macro avg       0.80      0.77      0.78      9769
weighted avg       0.84      0.85      0.85      9769



In [241]:
y_train_pred=best_model.predict_proba(x_train_filtered)[:,1]
treshold=0.7
y_pred_custom2=(y_train_pred>=treshold).astype(int)
print(classification_report(y_train,y_pred_custom2))

              precision    recall  f1-score   support

           0       0.90      0.94      0.92     17310
           1       0.78      0.67      0.72      5482

    accuracy                           0.87     22792
   macro avg       0.84      0.80      0.82     22792
weighted avg       0.87      0.87      0.87     22792

